In [11]:
# Create an API client
from anthropic import Anthropic
 
client = Anthropic()
model = "claude-sonnet-4-6"

# Functions to send messages and save context with Claude API

def add_user_message(messages, text):
    user_message = {"role": "user", "content" : text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content" : text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None, response_schema=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature" : temperature,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if response_schema:
        params["output_config"] = {
            "format": {
                "type": "json_schema",
                "schema": response_schema
            }
        }
    
    # Make request
    message = client.messages.create(**params)
    return message.content[0].text

In [7]:
messages = []

add_user_message(
    messages,
    "Generate a very short EventBridge rule as JSON. "
    "Respond with ONLY the raw JSON object — no markdown code fences, "
    "no ```json, no explanation, no leading or trailing text."
)

text = chat(messages)

text = text.strip()
if text.startswith("```"):
    text = text.split("```")[1]
    if text.startswith("json"):
        text = text[4:]
    text = text.strip()

print(text)

{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["stopped"]
  }
}


In [13]:
messages = []

add_user_message(messages, "Generate a very short EventBridge rule.")

schema = {
    "type": "object",
    "properties": {
        "source": {"type": "array", "items": {"type": "string"}},
        "detail-type": {"type": "array", "items": {"type": "string"}},
        "detail": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },
    "required": ["source", "detail-type", "detail"],
    "additionalProperties": False
}

text = chat(messages, response_schema=schema)
text

'{"source":["aws.ec2"],"detail-type":["EC2 Instance State-change Notification"],"detail":{}}'